# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Connect to Azure AI Foundry Agent Service using the **Microsoft Agent Framework**.
2. Give the agent a **tool** — a plain Python function it can call.
3. Run the agent and inspect its response.
4. Stream the agent's response token-by-token.

## Setup

Before running this notebook, make sure you have:

1. **An Azure AI Foundry project** with a deployed chat model (e.g. `gpt-4o-mini`).
2. **Logged in with the Azure CLI** — run `az login` in your terminal.
3. **Set the required environment variables:**
   - `AZURE_AI_PROJECT_ENDPOINT` — your Azure AI Foundry project endpoint.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.

The cell below installs the Python packages you need.

In [3]:
uv pip install agent-framework azure-ai-projects azure-identity -q

Note: you may need to restart the kernel to use updated packages.


c:\Users\jatsha01\ai-agents-for-beginners\.venv\Scripts\python.exe: No module named uv


Cell 1 : Setup Client and Imports

In [4]:
import os
import logging
from dotenv import load_dotenv

from azure.identity import AzureCliCredential
from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient

logging.getLogger("agent_framework").setLevel(logging.ERROR)

load_dotenv()

client = FoundryChatClient(
    credential=AzureCliCredential(),
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
)

## Creating Your First Agent

An agent needs two things:

- **Instructions** that tell it *who it is* and *how to behave* (a system prompt).
- **Tools** — Python functions decorated with `@tool` that the agent can call to retrieve information or perform actions.

Below we define a simple tool that returns a list of popular vacation destinations. The agent will use this tool when a user asks for travel recommendations.

approval_mode="never_require" means the agent can call this tool without pausing for human approval. This is okay here because the function only returns a list. For tools that send emails, delete files, update systems, or spend money, use approval instead. Microsoft’s docs describe approvals as the human-in-the-loop pattern for function execution.

In [6]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [7]:
travel_agent = Agent(
    client=client,
    name="TravelAgent",
    instructions="""
    You are a helpful travel planning assistant.
    When the user asks for vacation ideas, use the get_destinations tool.
    Recommend destinations with short reasons.
    Keep the response concise and practical.
    """,
    tools=[get_destinations],
)

Run the Agent

In [8]:
result = await travel_agent.run(
    "Suggest 3 vacation destinations for a 5-day trip. Give a short reason for each."
)

print(result.text if hasattr(result, "text") else result)

Here are 3 good options for a 5-day trip:

1. **Barcelona** — Great for a short break with beaches, architecture, and excellent food all close together.  
2. **Paris** — Ideal for 5 days if you want iconic sights, museums, cafés, and easy city exploring.  
3. **New York City** — A strong choice for a quick trip with Broadway, neighborhoods, shopping, and major landmarks packed into one city.

If you want, I can also suggest 3 options based on a vibe like **relaxing**, **romantic**, or **budget-friendly**.


## Streaming Responses

For a more interactive experience you can **stream** the agent's response. Instead of waiting for the full reply, the agent yields text chunks as they are generated. This is especially useful in chat interfaces where you want to display output in real time.

In [10]:
async for chunk in travel_agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

Tokyo is a great travel destination if you want a mix of ultra-modern city life and traditional culture.

Why people love it:
- Food: outstanding sushi, ramen, izakaya, street snacks, and convenience store food.
- Neighborhoods: Shibuya for energy, Shinjuku for nightlife, Asakusa for old Tokyo, Harajuku for fashion.
- Culture: temples, shrines, gardens, museums, anime and gaming districts like Akihabara.
- Transport: very efficient trains make it easy to explore.
- Safety: generally very safe, clean, and visitor-friendly.

Best for:
- Food lovers
- First-time Japan travelers
- Shoppers
- Pop culture fans
- Travelers who like busy cities

Things to keep in mind:
- It can be expensive, especially hotels.
- Stations can feel overwhelming at first.
- Summer is hot and humid; spring and autumn are usually easiest for most visitors.

Good times to visit:
- March to May for cherry blossom season
- October to November for pleasant weather
- Winter for fewer crowds and clear days

A simple firs

## Summary

In this lesson you learned how to:

- **Create a provider** that connects to Azure AI Foundry Agent Service via `AzureAIProjectAgentProvider`.
- **Define a tool** using the `@tool` decorator so the agent can call your Python functions.
- **Run the agent** with a user message and print its response.
- **Stream responses** for real-time output.

In the next lesson we will explore agentic frameworks in more depth and learn how to give agents more powerful tools and multi-step reasoning capabilities.